---
embed-resources: True
format:
    html:
        page-layout: full
        fontsize: 18px

title: 'Parte II | Markov y Monte Carlo'
author: 'Lorenzo Marquesini'
warning: False
toc: False
---

## Serie histórica

160 semanas para estimar la cadena. Columnas `period` y `state`, en el orden Normal, Pico, Disrupcion. La última semana es Disrupcion: de ahí arranca la simulación. La semilla es 20260816.


In [58]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.observaciones_markov import ESTADOS, SEMILLA, observaciones_historicas

DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"

observaciones = observaciones_historicas(seed=SEMILLA)
observaciones.to_csv(DATA / "markov_observations.csv", index=False)

destinations = pd.read_csv(DATA / "destinations.csv")
routes = pd.read_csv(DATA / "routes.csv")
origins = pd.read_csv(DATA / "origins.csv")

salidas = {estado: 0 for estado in ESTADOS}
for origen, _destino in zip(observaciones["state"], observaciones["state"].iloc[1:]):
    salidas[origen] += 1

assert list(observaciones.columns) == ["period", "state"]
assert len(observaciones) == 160
assert observaciones["state"].iloc[-1] == "Disrupcion"
assert all(salidas[estado] > 0 for estado in ESTADOS)

print("Semilla:", SEMILLA)
print("Último estado:", observaciones["state"].iloc[-1])
print("Salidas observadas:", salidas)
display(observaciones.head())
display(destinations)
display(routes.head())
display(origins)


Semilla: 20260816
Último estado: Disrupcion
Salidas observadas: {'Normal': 83, 'Pico': 66, 'Disrupcion': 10}


,period,state
0,0,Normal
1,1,Normal
2,2,Pico
3,3,Pico
4,4,Pico


,destination,nominal_demand,shortage_penalty,initial_inventory,storage_capacity,holding_cost,waste_cost
0,Capital,50.0,80.0,8.0,65.0,0.40,0.80
1,Oeste,42.0,70.0,6.0,55.0,0.35,0.70
2,Este,38.0,65.0,5.0,50.0,0.32,0.65
3,Costa,46.0,75.0,7.0,60.0,0.38,0.75
4,Interior,35.0,60.0,5.0,48.0,0.30,0.60


,route_id,origin,destination,unit_cost,capacity
0,N-CAP,Norte,Capital,4.0,42.0
1,N-OES,Norte,Oeste,6.0,28.0
2,N-EST,Norte,Este,5.0,30.0
3,N-COS,Norte,Costa,9.0,18.0
4,N-INT,Norte,Interior,7.0,26.0


,origin,supply
0,Norte,60.0
1,Centro,95.0
2,Sur,75.0


In [59]:
observaciones

,period,state
0,0,Normal
1,1,Normal
2,2,Pico
3,3,Pico
4,4,Pico
...,...,...
155,155,Pico
156,156,Normal
157,157,Normal
158,158,Disrupcion


## Cadena de Markov

Tres estados:
* Normal
* Pico
* Disrupción

In [60]:
from IPython.display import HTML

BLUE = "#1D24AB"
RED = "#C0392B"
INK = "#1F2430"
LINE = "#D5D8E2"
ETIQUETAS = {"Normal": "Normal", "Pico": "Pico", "Disrupcion": "Disrupción"}

def estimar_matriz(serie, estados=ESTADOS):
    origen = serie["state"].iloc[:-1].to_numpy()
    destino = serie["state"].iloc[1:].to_numpy()
    conteos = (
        pd.crosstab(origen, destino)
        .reindex(index=list(estados), columns=list(estados), fill_value=0)
        .astype(int)
    )
    salidas = conteos.sum(axis=1)
    vacias = salidas[salidas == 0].index.tolist()
    if vacias:
        raise RuntimeError(
            "Fila no estimable, sin salidas observadas: " + ", ".join(map(str, vacias))
        )
    matriz = conteos.div(salidas, axis=0)
    if (matriz < 0).any().any() or not (matriz.sum(axis=1) - 1).abs().max() < 1e-9:
        raise RuntimeError("La matriz estimada no es estocástica por filas")
    return conteos, matriz

conteos, matriz = estimar_matriz(observaciones)

def td(text, color=INK, weight="500", italic=False):
    style = (
        f"border:1px solid {LINE}; padding:8px 16px; text-align:center; background:#fff;"
        f"color:{color}; font-weight:{weight};"
    )
    if italic:
        style += "font-style:italic;"
    return f"<td style='{style}'>{text}</td>"

def th(text, color, italic=True):
    return td(text, color=color, weight="700", italic=italic).replace("td", "th", 2)

corner = (
    f"<th style='position:relative; min-width:118px; height:56px; border:1px solid {LINE};"
    "background:linear-gradient(to top right,#fff calc(50% - 0.7px),#C5C8D4 calc(50% - 0.7px),"
    f"#C5C8D4 calc(50% + 0.7px),#fff calc(50% + 0.7px));'>"
    f"<span style='position:absolute; top:6px; right:8px; color:{RED}; font-style:italic;'>Hacia</span>"
    f"<span style='position:absolute; bottom:6px; left:8px; color:{BLUE}; font-style:italic;'>Desde</span>"
    "</th>"
)
head = "".join(th(ETIQUETAS[estado], RED) for estado in matriz.columns)
body = []
for estado, fila in matriz.iterrows():
    celdas = th(ETIQUETAS[estado], BLUE) + "".join(td(f"{valor:.2f}") for valor in fila)
    body.append(f"<tr>{celdas}</tr>")

tabla = (
    "<table style='border-collapse:collapse; font-family:sans-serif; font-size:15px;'>"
    f"<thead><tr>{corner}{head}</tr></thead>"
    f"<tbody>{''.join(body)}</tbody></table>"
)
HTML(tabla)


HaciaDesde,Normal,Pico,Disrupción
Normal,0.72,0.24,0.04
Pico,0.24,0.68,0.08
Disrupción,0.60,0.10,0.30


### Simular posibles futuros

Una trayectoria tiene 26 semanas y empieza en Disrupción. El estado de la semana fija la media de la demanda, la probabilidad de que una ruta opere y el multiplicador de capacidad. Dado el estado, demandas y disponibilidades se sortean por separado.

#### 1. Las 26 semanas

La semana 1 es Disrupción, el último estado de la serie. De la semana 2 en adelante, el estado se sortea con la fila correspondiente de la matriz estimada.

In [61]:
import numpy as np

HORIZONTE = 26

def semanas(rng, matriz, horizonte=HORIZONTE, inicial="Disrupcion"):
    nombres = list(matriz.columns)
    estados = [inicial]
    for _ in range(horizonte - 1):
        fila = matriz.loc[estados[-1]].to_numpy(dtype=float)
        fila = fila / fila.sum()
        estados.append(nombres[int(rng.choice(len(nombres), p=fila))])
    return pd.DataFrame({"semana": np.arange(1, horizonte + 1), "estado": estados})

rng = np.random.default_rng(SEMILLA)
trayectoria = semanas(rng, matriz)
assert len(trayectoria) == HORIZONTE
assert trayectoria["estado"].iloc[0] == "Disrupcion"
trayectoria


,semana,estado
0,1,Disrupcion
1,2,Normal
2,3,Pico
3,4,Pico
4,5,Pico
5,6,Pico
6,7,Pico
7,8,Pico
8,9,Pico
9,10,Pico


#### 2. Demanda

En cada zona, la demanda de la semana es Poisson. La media es la demanda nominal multiplicada por el `demand_multiplier` del estado. Las cinco zonas se sortean independientes.

In [62]:
def demanda(rng, demanda_nominal, multiplicador):
    media = np.asarray(demanda_nominal, dtype=float) * multiplicador
    return rng.poisson(media)


#### 3. Disponibilidad

Cada ruta está disponible o no. El indicador es Bernoulli con probabilidad `route_availability` del estado. Las quince rutas se sortean independientes, también de la demanda.

In [63]:
def disponibilidad(rng, n_rutas, probabilidad):
    return (rng.random(n_rutas) < probabilidad).astype(int)


#### 4. Capacidad efectiva

No es otro sorteo. Si la ruta no está disponible, la capacidad es cero. Si lo está, la capacidad nominal se multiplica por el `capacity_multiplier` del estado.

In [64]:
def capacidad_efectiva(disponible, capacidad_nominal, multiplicador):
    operativo = np.asarray(disponible, dtype=float)
    nominal = np.asarray(capacidad_nominal, dtype=float)
    return operativo * nominal * multiplicador


### Operación logística

El plan queda fijo durante las 26 semanas. El futuro no depende de la política: estado, demanda y disponibilidad se generan una vez y las dos políticas recorren ese mismo futuro.

En cada semana el algoritmo hace siete pasos.

1. Toma el estado y genera la demanda de cada zona y la disponibilidad de cada ruta.
2. De cada envío planificado, transporta solo lo que entra en la capacidad efectiva.
3. Suma lo entregado al inventario que la zona ya tenía.
4. Atiende la mayor demanda posible.
5. Registra el faltante. Esa demanda se pierde: no queda como pedido para la semana siguiente.
6. Guarda el sobrante hasta el límite de almacenamiento de la zona.
7. Descarta el exceso y suma el costo de la semana: transporte, penalización, almacenamiento y descarte.

In [65]:
def preparar_semana(rng, estado, rutas, zonas, parametros):
    fila = parametros.loc[estado]
    pedida = demanda(rng, zonas["nominal_demand"].to_numpy(), float(fila["demand_multiplier"]))
    abierta = disponibilidad(rng, len(rutas), float(fila["route_availability"]))
    tope = capacidad_efectiva(abierta, rutas["capacity"].to_numpy(), float(fila["capacity_multiplier"]))
    return pedida, abierta, tope


def entorno(rng, matriz, rutas, zonas, parametros, horizonte=HORIZONTE, inicial="Disrupcion"):
    nombres = list(matriz.columns)
    estado = inicial
    estados, pedidas, abiertas, topes = [], [], [], []
    for t in range(horizonte):
        if t:
            fila = matriz.loc[estado].to_numpy(dtype=float)
            fila = fila / fila.sum()
            estado = nombres[int(rng.choice(len(nombres), p=fila))]
        pedida, abierta, tope = preparar_semana(rng, estado, rutas, zonas, parametros)
        estados.append(estado)
        pedidas.append(np.asarray(pedida, dtype=float))
        abiertas.append(np.asarray(abierta, dtype=int))
        topes.append(np.asarray(tope, dtype=float))
    return {
        "estado": estados,
        "demanda": np.vstack(pedidas),
        "disponible": np.vstack(abiertas),
        "capacidad": np.vstack(topes),
    }


def operar_semana(plan, rutas, zonas, pedida, capacidad, inventario):
    envio = rutas[["route_id", "origin", "destination", "unit_cost"]].copy()
    flujo = plan.set_index("route_id")["planned_flow"]
    if envio["route_id"].isin(flujo.index).sum() != len(envio):
        raise RuntimeError("El plan no tiene un flujo para cada ruta")
    envio["planificado"] = envio["route_id"].map(flujo).astype(float)
    envio["capacidad_efectiva"] = np.asarray(capacidad, dtype=float)
    envio["despachado"] = np.minimum(envio["planificado"], envio["capacidad_efectiva"])

    zonas_idx = zonas.set_index("destination")
    cuadro = pd.DataFrame(index=zonas_idx.index)
    cuadro["demanda"] = pd.Series(np.asarray(pedida, dtype=float), index=zonas["destination"].to_numpy())
    cuadro["inventario_inicial"] = inventario.reindex(cuadro.index).astype(float)
    cuadro["recepcion"] = envio.groupby("destination")["despachado"].sum().reindex(cuadro.index).fillna(0.0)
    cuadro["existencias"] = cuadro["inventario_inicial"] + cuadro["recepcion"]
    cuadro["atendido"] = np.minimum(cuadro["existencias"], cuadro["demanda"])
    cuadro["faltante"] = cuadro["demanda"] - cuadro["atendido"]
    cuadro["sobrante"] = cuadro["existencias"] - cuadro["atendido"]
    cuadro["inventario"] = np.minimum(cuadro["sobrante"], zonas_idx["storage_capacity"])
    cuadro["descarte"] = np.maximum(cuadro["sobrante"] - zonas_idx["storage_capacity"], 0.0)

    if (cuadro[["demanda", "recepcion", "atendido", "faltante", "inventario", "descarte"]] < -1e-9).any().any():
        raise RuntimeError("Apareció un valor negativo en el balance semanal")
    if not np.allclose(cuadro["atendido"] + cuadro["faltante"], cuadro["demanda"]):
        raise RuntimeError("La demanda atendida más el faltante no cubre la demanda")
    if not np.allclose(cuadro["atendido"] + cuadro["sobrante"], cuadro["existencias"]):
        raise RuntimeError("Lo atendido más el sobrante no coincide con las existencias")
    if not np.allclose(cuadro["inventario"] + cuadro["descarte"], cuadro["sobrante"]):
        raise RuntimeError("Inventario más descarte no coincide con el sobrante")

    costo_transporte = float((envio["despachado"] * envio["unit_cost"]).sum())
    costo_faltante = float((cuadro["faltante"] * zonas_idx["shortage_penalty"]).sum())
    costo_almacen = float((cuadro["inventario"] * zonas_idx["holding_cost"]).sum())
    costo_descarte = float((cuadro["descarte"] * zonas_idx["waste_cost"]).sum())
    costos = pd.Series({
        "transporte": costo_transporte,
        "faltante": costo_faltante,
        "almacen": costo_almacen,
        "descarte": costo_descarte,
        "total": costo_transporte + costo_faltante + costo_almacen + costo_descarte,
    })
    return envio, cuadro, costos, cuadro["inventario"].copy()

def operar(plan, rutas, zonas, futuro):
    inventario = zonas.set_index("destination")["initial_inventory"].astype(float)
    filas = []
    for t, estado in enumerate(futuro["estado"]):
        _envio, cuadro, costos, inventario = operar_semana(
            plan, rutas, zonas, futuro["demanda"][t], futuro["capacidad"][t], inventario
        )
        filas.append({
            "semana": t + 1,
            "estado": estado,
            "demanda": float(cuadro["demanda"].sum()),
            "atendido": float(cuadro["atendido"].sum()),
            "faltante": float(cuadro["faltante"].sum()),
            "descarte": float(cuadro["descarte"].sum()),
            "inventario": float(inventario.sum()),
            "costo": float(costos["total"]),
        })
    resumen = pd.DataFrame(filas)
    resumen.attrs["servicio"] = float(resumen["atendido"].sum() / resumen["demanda"].sum())
    return resumen


### Control de una semana

Semana fija, no sorteada: la demanda es la nominal, todas las rutas están abiertas y la capacidad efectiva es la nominal. El plan es un óptimo de la Parte I. Comprueba el balance antes de usar los multiplicadores de cada estado, que esta sección recibe y no asigna.

In [66]:
plan_nominal = pd.DataFrame({
    "route_id": [
        "N-CAP", "N-OES", "N-EST", "N-COS", "N-INT",
        "C-CAP", "C-OES", "C-EST", "C-COS", "C-INT",
        "S-CAP", "S-OES", "S-EST", "S-COS", "S-INT",
    ],
    "planned_flow": [42, 0, 4, 0, 0, 8, 35, 34, 8, 5, 0, 7, 0, 38, 30],
})

inventario_inicial = destinations.set_index("destination")["initial_inventory"].astype(float)
pedida_nominal = destinations["nominal_demand"].to_numpy()
capacidad_nominal = routes["capacity"].to_numpy()

envio, cuadro, costos, inventario_final = operar_semana(
    plan_nominal, routes, destinations, pedida_nominal, capacidad_nominal, inventario_inicial
)

assert np.allclose(cuadro["faltante"], 0)
assert np.allclose(cuadro["descarte"], 0)
assert np.allclose(inventario_final, inventario_inicial)
assert np.allclose(costos["transporte"], 818)
assert np.allclose(
    costos["almacen"],
    float((inventario_inicial * destinations.set_index("destination")["holding_cost"]).sum()),
)

display(cuadro[["demanda", "recepcion", "atendido", "faltante", "inventario", "descarte"]])
costos


,demanda,recepcion,atendido,faltante,inventario,descarte
destination,,,,,,
Capital,50.0,50.0,50.0,0.0,8.0,0.0
Oeste,42.0,42.0,42.0,0.0,6.0,0.0
Este,38.0,38.0,38.0,0.0,5.0,0.0
Costa,46.0,46.0,46.0,0.0,7.0,0.0
Interior,35.0,35.0,35.0,0.0,5.0,0.0


transporte    818.00
faltante        0.00
almacen        11.06
descarte        0.00
total         829.06
dtype: float64

### Las dos políticas

Las dos políticas salen de la Parte I y no se vuelven a optimizar. La nominal cubre la demanda de una semana típica. El resguardo se calculó con una demanda de diseño 5 % mayor, redondeada hacia arriba. En cada futuro las dos enfrentan la misma demanda, las mismas rutas y el mismo inventario inicial.

In [67]:
plan_resguardo = pd.DataFrame({
    "route_id": [
        "N-CAP", "N-OES", "N-EST", "N-COS", "N-INT",
        "C-CAP", "C-OES", "C-EST", "C-COS", "C-INT",
        "S-CAP", "S-OES", "S-EST", "S-COS", "S-INT",
    ],
    "planned_flow": [42, 3, 9, 0, 0, 11, 35, 31, 11, 7, 0, 7, 0, 38, 30],
})

def costo_del_plan(plan):
    tabla = plan.merge(routes, on="route_id")
    return float((tabla["planned_flow"] * tabla["unit_cost"]).sum())

uso_resguardo = plan_resguardo.merge(routes, on="route_id").groupby("origin")["planned_flow"].sum()
oferta = origins.set_index("origin")["supply"]
assert np.isclose(costo_del_plan(plan_nominal), 818)
assert np.isclose(costo_del_plan(plan_resguardo), 892)
assert np.isclose(plan_resguardo["planned_flow"].sum(), 224)
assert (uso_resguardo <= oferta.loc[uso_resguardo.index] + 1e-8).all()

pd.DataFrame({
    "política": ["Nominal", "Resguardo"],
    "flujo planificado": [plan_nominal["planned_flow"].sum(), plan_resguardo["planned_flow"].sum()],
    "costo del plan": [costo_del_plan(plan_nominal), costo_del_plan(plan_resguardo)],
})


,política,flujo planificado,costo del plan
0,Nominal,211,818.0
1,Resguardo,224,892.0


### Experimento Monte Carlo

La campaña tiene 1.200 trayectorias de 26 semanas. La semilla maestra es 20260816: cada réplica recibe un generador propio, así el experimento se puede repetir. Las primeras 200 son el piloto y no se descartan; la comparación usa las 1.200.

Cada trayectoria se sortea una sola vez. Las dos políticas reciben los mismos estados, las mismas demandas y las mismas disponibilidades.

Los multiplicadores no estaban en el enunciado. La tabla sigue la descripción de cada estado: Normal queda en el nivel habitual, con capacidad completa y casi todas las rutas abiertas; Pico sube la demanda y baja la capacidad; Disrupción mantiene la demanda y recorta con fuerza capacidad y disponibilidad.

In [68]:
from scipy.stats import t as student_t

N_PILOTO = 200
N_CAMPANA = 1200
COLUMNAS_ESTADO = ["demand_multiplier", "capacity_multiplier", "route_availability"]

def cargar_parametros(ruta):
    if not ruta.exists():
        raise FileNotFoundError(
            "Falta data/markov_state_parameters.csv. "
            "Hacen falta demand_multiplier, capacity_multiplier y route_availability "
            "para Normal, Pico y Disrupcion."
        )
    tabla = pd.read_csv(ruta)
    if "state" not in tabla.columns:
        raise ValueError("markov_state_parameters.csv necesita la columna state")
    tabla = tabla.set_index("state").loc[list(ESTADOS), COLUMNAS_ESTADO].astype(float)
    if (tabla < 0).any().any() or (tabla["route_availability"] > 1).any():
        raise ValueError("Probabilidad fuera de [0, 1] o multiplicador negativo")
    return tabla

def flujo_de(plan, rutas):
    flujo = plan.set_index("route_id")["planned_flow"]
    if not set(rutas["route_id"]) <= set(flujo.index):
        raise RuntimeError("El plan no cubre todas las rutas")
    return rutas["route_id"].map(flujo).to_numpy(dtype=float)

def red_logistica(rutas, zonas):
    indice = {zona: i for i, zona in enumerate(zonas["destination"])}
    return {
        "zona": rutas["destination"].map(indice).to_numpy(),
        "costo": rutas["unit_cost"].to_numpy(dtype=float),
        "penalizacion": zonas["shortage_penalty"].to_numpy(dtype=float),
        "almacenaje": zonas["holding_cost"].to_numpy(dtype=float),
        "descarte": zonas["waste_cost"].to_numpy(dtype=float),
        "capacidad_deposito": zonas["storage_capacity"].to_numpy(dtype=float),
        "inventario_inicial": zonas["initial_inventory"].to_numpy(dtype=float),
    }

def evaluar_plan(flujo, futuro, red):
    n_zonas = len(red["inventario_inicial"])
    inventario = red["inventario_inicial"].copy()
    costo = atendido = pedida = faltante = descarte = stock = 0.0
    semanas_con_faltante = 0
    for t in range(len(futuro["estado"])):
        enviado = np.minimum(flujo, futuro["capacidad"][t])
        recepcion = np.zeros(n_zonas)
        np.add.at(recepcion, red["zona"], enviado)
        demanda_t = futuro["demanda"][t]
        existencias = inventario + recepcion
        cubierto = np.minimum(existencias, demanda_t)
        perdido = demanda_t - cubierto
        sobrante = existencias - cubierto
        inventario = np.minimum(sobrante, red["capacidad_deposito"])
        tirado = np.maximum(sobrante - red["capacidad_deposito"], 0.0)
        costo += float(
            enviado @ red["costo"]
            + perdido @ red["penalizacion"]
            + inventario @ red["almacenaje"]
            + tirado @ red["descarte"]
        )
        atendido += float(cubierto.sum())
        pedida += float(demanda_t.sum())
        faltante += float(perdido.sum())
        descarte += float(tirado.sum())
        stock += float(inventario.sum())
        if perdido.sum() > 1e-9:
            semanas_con_faltante += 1
    horizonte = len(futuro["estado"])
    return {
        "costo": costo,
        "servicio": atendido / pedida if pedida > 0 else np.nan,
        "faltante": faltante,
        "probabilidad_faltante": float(semanas_con_faltante > 0),
        "fraccion_semanas_faltante": semanas_con_faltante / horizonte,
        "inventario": stock / horizonte,
        "descarte": descarte,
    }

def campana(generadores, matriz, rutas, zonas, parametros, flujo_nominal, flujo_resguardo):
    red = red_logistica(rutas, zonas)
    filas = []
    estados = []
    for i, rng in enumerate(generadores):
        futuro = entorno(rng, matriz, rutas, zonas, parametros)
        nominal = evaluar_plan(flujo_nominal, futuro, red)
        resguardo = evaluar_plan(flujo_resguardo, futuro, red)
        fila = {"replica": i, "demanda": float(futuro["demanda"].sum())}
        fila.update({f"{clave}_nominal": valor for clave, valor in nominal.items()})
        fila.update({f"{clave}_resguardo": valor for clave, valor in resguardo.items()})
        filas.append(fila)
        estados.append(futuro["estado"])
    return pd.DataFrame(filas), estados

parametros = cargar_parametros(DATA / "markov_state_parameters.csv")
display(parametros)
flujo_nominal = flujo_de(plan_nominal, routes)
flujo_resguardo = flujo_de(plan_resguardo, routes)
generadores = [
    np.random.default_rng(hijo)
    for hijo in np.random.SeedSequence(SEMILLA).spawn(N_CAMPANA)
]
resultados, estados_campana = campana(
    generadores, matriz, routes, destinations, parametros, flujo_nominal, flujo_resguardo
)
assert len(resultados) == N_CAMPANA
assert resultados["replica"].is_monotonic_increasing
resultados.head()


,demand_multiplier,capacity_multiplier,route_availability
state,,,
Normal,1.0,1.0,0.95
Pico,1.2,0.8,0.90
Disrupcion,1.0,0.5,0.50


,replica,demanda,costo_nominal,servicio_nominal,faltante_nominal,probabilidad_faltante_nominal,fraccion_semanas_faltante_nominal,inventario_nominal,descarte_nominal,costo_resguardo,servicio_resguardo,faltante_resguardo,probabilidad_faltante_resguardo,fraccion_semanas_faltante_resguardo,inventario_resguardo,descarte_resguardo
0,0,5760.0,112188.856,0.765538,1350.5,1.0,0.961538,21.476923,0.0,98474.430,0.803906,1129.5,1.0,0.576923,50.100000,11.0
1,1,5761.0,136595.974,0.708488,1679.4,1.0,0.961538,10.992308,0.0,115969.306,0.763687,1361.4,1.0,0.923077,20.630769,0.0
2,2,5855.0,88627.950,0.833168,976.8,1.0,0.884615,30.807692,0.0,70821.090,0.880478,699.8,1.0,0.692308,56.153846,16.0
3,3,5699.0,90542.744,0.818968,1031.7,1.0,0.961538,22.761538,0.0,70229.186,0.874662,714.3,1.0,0.807692,47.761538,6.0
4,4,5666.0,100964.105,0.799135,1138.1,1.0,0.961538,18.288462,0.0,83098.062,0.847812,862.3,1.0,0.807692,37.953846,0.0


### Piloto de 200

Las primeras 200 trayectorias revisan que el modelo no produzca valores imposibles, que la semilla repita la primera réplica y qué estados aparecen. Siguen formando parte de las 1.200. El piloto no se usa para cambiar métricas ni para elegir la política.

In [69]:
def chequear(tabla):
    assert tabla["servicio_nominal"].between(0, 1).all()
    assert tabla["servicio_resguardo"].between(0, 1).all()
    for politica in ("nominal", "resguardo"):
        assert (tabla[f"costo_{politica}"] >= -1e-8).all()
        assert (tabla[f"faltante_{politica}"] >= -1e-8).all()
        assert (tabla[f"descarte_{politica}"] >= -1e-8).all()
        assert (tabla[f"inventario_{politica}"] >= -1e-8).all()
        assert tabla[f"fraccion_semanas_faltante_{politica}"].between(0, 1).all()
        assert set(tabla[f"probabilidad_faltante_{politica}"].unique()) <= {0.0, 1.0}
    assert (tabla["demanda"] > 0).all()

piloto = resultados.iloc[:N_PILOTO].copy()
chequear(piloto)
chequear(resultados)

hijo_cero = np.random.SeedSequence(SEMILLA).spawn(N_CAMPANA)[0]
futuro_cero = entorno(np.random.default_rng(hijo_cero), matriz, routes, destinations, parametros)
red = red_logistica(routes, destinations)
repetida_nominal = evaluar_plan(flujo_nominal, futuro_cero, red)
repetida_resguardo = evaluar_plan(flujo_resguardo, futuro_cero, red)
assert np.isclose(repetida_nominal["costo"], resultados.loc[0, "costo_nominal"])
assert np.isclose(repetida_resguardo["costo"], resultados.loc[0, "costo_resguardo"])
assert futuro_cero["estado"][0] == "Disrupcion"

ocupacion = pd.Series(
    [estado for trayectoria in estados_campana[:N_PILOTO] for estado in trayectoria],
    name="semanas",
)
escala = pd.Series({
    "costo nominal": piloto["costo_nominal"].mean(),
    "costo resguardo": piloto["costo_resguardo"].mean(),
    "servicio nominal": piloto["servicio_nominal"].mean(),
    "servicio resguardo": piloto["servicio_resguardo"].mean(),
})
print("Réplicas del piloto:", len(piloto), "de", len(resultados))
print("Invariantes y repetición de la primera réplica: ok")
display(ocupacion.value_counts().rename("semanas").to_frame())
escala


Réplicas del piloto: 200 de 1200
Invariantes y repetición de la primera réplica: ok


,semanas
semanas,
Normal,2682
Pico,1915
Disrupcion,603


costo nominal         123016.512055
costo resguardo       103260.380000
servicio nominal           0.748063
servicio resguardo         0.800181
dtype: float64

### Comparación de políticas

La diferencia de cada réplica es resguardo menos nominal. Una diferencia de costo negativa favorece al resguardo; una de servicio positiva también. La media, el error estándar y el intervalo del 95 % usan la t de Student con 1.199 grados de libertad. El intervalo mide la precisión de esa ventaja, no solo cuál política quedó adelante en el promedio.

El inventario medio es el stock de cierre, sumado entre zonas y promediado en las 26 semanas. El descarte es el total de unidades tiradas en la trayectoria.

In [70]:
def intervalo(diferencia):
    n = len(diferencia)
    media = float(np.mean(diferencia))
    error = float(np.std(diferencia, ddof=1) / np.sqrt(n))
    critico = float(student_t.ppf(0.975, n - 1))
    return media, error, media - critico * error, media + critico * error

metricas = [
    ("costo", "Costo total"),
    ("servicio", "Nivel de servicio"),
    ("faltante", "Unidades faltantes"),
    ("probabilidad_faltante", "Probabilidad de algún faltante"),
    ("fraccion_semanas_faltante", "Fracción de semanas con faltante"),
    ("inventario", "Inventario medio"),
    ("descarte", "Descarte"),
]
filas = []
for clave, nombre in metricas:
    diferencia = resultados[f"{clave}_resguardo"] - resultados[f"{clave}_nominal"]
    media, error, bajo, alto = intervalo(diferencia.to_numpy())
    filas.append({
        "metrica": nombre,
        "nominal": float(resultados[f"{clave}_nominal"].mean()),
        "resguardo": float(resultados[f"{clave}_resguardo"].mean()),
        "diferencia": media,
        "error_estandar": error,
        "ic_inferior": bajo,
        "ic_superior": alto,
    })
comparacion = pd.DataFrame(filas)

OUTPUTS.mkdir(parents=True, exist_ok=True)
matriz_out = matriz.copy()
matriz_out.index.name = "state"
matriz_out.columns.name = "next"
matriz_out.to_csv(OUTPUTS / "markov_matrix.csv")
resultados.to_csv(OUTPUTS / "montecarlo_summary.csv", index=False)
comparacion.to_csv(OUTPUTS / "paired_comparison.csv", index=False)
comparacion.round(4)


,metrica,nominal,resguardo,diferencia,error_estandar,ic_inferior,ic_superior
0,Costo total,124630.4726,104673.0691,-19957.4034,58.1406,-20071.4721,-19843.3347
1,Nivel de servicio,0.7441,0.7966,0.0525,0.0001,0.0523,0.0528
2,Unidades faltantes,1516.6064,1206.3560,-310.2504,0.8388,-311.8962,-308.6047
3,Probabilidad de algún faltante,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000
4,Fracción de semanas con faltante,0.9485,0.8521,-0.0965,0.0023,-0.1009,-0.0920
5,Inventario medio,12.6016,26.0108,13.4092,0.2501,12.9186,13.8998
6,Descarte,0.0258,1.0916,1.0658,0.1379,0.7951,1.3364
